# Reviewing extracted pipeline results

Loads every `src/pharmas/*/*.parquet` file and checks:

1. Schema — columns, dtypes, union across companies
2. Validity — required fields, `phase` enum, `trial_id`/`source_url`/`extraction_date` formats
3. Sanity — duplicates, empty strings, `company` vs. folder name, phase aliases
4. Coverage — rows/company, phase distribution, missing-field rates
5. Conclusions

Reproducible: re-run top to bottom, no manual steps. Only reads from `src/pharmas/`, writes nothing.

In [1]:
import re
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PARQUET_GLOB = "src/pharmas/*/*.parquet"

paths = sorted(REPO_ROOT.glob(PARQUET_GLOB))
assert paths, f"No parquet files found under {REPO_ROOT / PARQUET_GLOB}"

frames = []
for p in paths:
    df = pd.read_parquet(p)
    df["_source_file"] = str(p.relative_to(REPO_ROOT))
    df["_source_folder"] = p.parent.name
    frames.append(df)

print(f"Found {len(paths)} parquet files:")
for p in paths:
    print(" -", p.relative_to(REPO_ROOT))

Found 9 parquet files:
 - src/pharmas/abbvie/abbvie_pipeline.parquet
 - src/pharmas/astrazeneca/astrazeneca_pipeline.parquet
 - src/pharmas/gsk/gsk_pipeline.parquet
 - src/pharmas/msd/msd_pipeline.parquet
 - src/pharmas/novartis/novartis_pipeline.parquet
 - src/pharmas/novonordisk/novonordisk_pipeline.parquet
 - src/pharmas/pfizer/pfizer_pipeline.parquet
 - src/pharmas/roche/roche_pipeline.parquet
 - src/pharmas/sanofi/sanofi_pipeline.parquet


## 1. Schema

Base schema (`src/schema.py`, `PipelineRecord`) requires: `company`, `asset_name`, `indication`, `phase`.
Optional: `synonyms`, `mechanism_of_action`, `therapeutic_area`, `trial_id`, `source_url`, `extraction_date`, `modality`, `notes`.
`model_config = {"extra": "allow"}` — extra columns (`others`) are permitted per-source extras.

In [2]:
REQUIRED_FIELDS = ["company", "asset_name", "indication", "phase"]
BASE_OPTIONAL_FIELDS = [
    "synonyms", "mechanism_of_action", "therapeutic_area", "trial_id",
    "source_url", "extraction_date", "modality", "notes",
]
KNOWN_EXTRA_FIELDS = ["others"]
META_COLS = {"_source_file", "_source_folder"}

schema_rows = []
for df, p in zip(frames, paths):
    cols = set(df.columns) - META_COLS
    schema_rows.append({
        "folder": p.parent.name,
        "n_rows": len(df),
        "columns": sorted(cols),
        "missing_required": sorted(set(REQUIRED_FIELDS) - cols),
        "unknown_extra": sorted(cols - set(REQUIRED_FIELDS) - set(BASE_OPTIONAL_FIELDS) - set(KNOWN_EXTRA_FIELDS)),
    })

schema_df = pd.DataFrame(schema_rows)
schema_df

,folder,n_rows,columns,missing_required,unknown_extra
0,abbvie,97,"[asset_name, company, extraction_date, indicat...",[],[]
1,astrazeneca,199,"[asset_name, company, extraction_date, indicat...",[],[]
2,gsk,76,"[asset_name, company, extraction_date, indicat...",[],[]
3,msd,109,"[asset_name, company, extraction_date, indicat...",[],[]
4,novartis,106,"[asset_name, company, extraction_date, indicat...",[],[]
5,novonordisk,37,"[asset_name, company, extraction_date, indicat...",[],[]
6,pfizer,96,"[asset_name, company, extraction_date, indicat...",[],[]
7,roche,128,"[asset_name, company, extraction_date, indicat...",[],[]
8,sanofi,77,"[asset_name, company, extraction_date, indicat...",[],[]


In [3]:
# Combine into one long dataframe (pandas aligns/union columns, filling absent ones with NaN)
data = pd.concat(frames, ignore_index=True, sort=False)
print(f"Combined: {len(data)} rows, {len(paths)} companies, {len(data.columns)} columns")

# dtype per column, per source file — flags a column stored inconsistently across companies
dtype_matrix = pd.DataFrame({
    p.parent.name: pd.read_parquet(p).dtypes.astype(str)
    for p in paths
}).T
dtype_matrix

Combined: 925 rows, 9 companies, 15 columns


,asset_name,company,extraction_date,indication,mechanism_of_action,modality,notes,others,phase,source_url,synonyms,therapeutic_area,trial_id
abbvie,str,str,str,str,str,str,str,object,str,str,object,str,object
astrazeneca,str,str,str,str,str,str,str,NaN,str,str,object,str,object
gsk,str,str,str,str,str,NaN,object,object,str,str,object,str,object
msd,str,str,object,str,str,NaN,str,object,str,str,object,str,str
novartis,str,str,str,str,str,object,object,object,str,str,object,str,object
novonordisk,str,str,str,str,str,NaN,object,object,str,str,object,str,object
pfizer,str,str,str,str,str,str,object,object,str,str,object,str,object
roche,str,str,str,str,str,NaN,object,object,str,str,object,str,object
sanofi,str,str,str,str,str,NaN,str,object,str,str,object,str,object


## 2. Validity

Required fields non-null/non-empty; `phase` in the controlled enum (`docs/data-model.md`); `trial_id` looks like `NCT########`; `source_url` looks like a URL; `extraction_date` parses and isn't in the future.

In [4]:
def is_blank(series):
    return series.isna() | (series.astype(str).str.strip() == "")

required_issues = []
for field in REQUIRED_FIELDS:
    if field not in data.columns:
        continue
    blank = is_blank(data[field])
    if blank.any():
        bad = data.loc[blank, ["_source_folder", "company", "asset_name", "indication", "phase"]]
        for _, row in bad.iterrows():
            required_issues.append({"issue": f"blank `{field}`", **row.to_dict()})

required_issues_df = pd.DataFrame(required_issues)
print(f"Rows with a blank required field: {len(required_issues_df)}")
required_issues_df

Rows with a blank required field: 0


""


In [5]:
# phase enum (src/schema.py Phase / docs/data-model.md)
VALID_PHASES = {
    "Preclinical", "Phase 1", "Phase 1/2", "Phase 2", "Phase 2/3",
    "Phase 3", "Preregistration", "Registered", "Discontinued",
}

bad_phase = data[~data["phase"].isin(VALID_PHASES)]
print(f"Rows with an out-of-enum `phase`: {len(bad_phase)}")
print("\nPhase value counts (all companies):")
print(data["phase"].value_counts(dropna=False))
bad_phase[["_source_folder", "company", "asset_name", "phase"]]

Rows with an out-of-enum `phase`: 0

Phase value counts (all companies):
phase
Phase 3            317
Phase 2            275
Phase 1            225
Preregistration     49
Registered          46
Discontinued        13
Name: count, dtype: int64


,_source_folder,company,asset_name,phase


In [6]:
# trial_id format: NCT + 8 digits
NCT_RE = re.compile(r"^NCT\d{8}$")
has_trial_id = data["trial_id"].notna() & (data["trial_id"].astype(str).str.strip() != "")
bad_trial_id = data[has_trial_id & ~data["trial_id"].astype(str).str.match(NCT_RE)]
print(f"trial_id populated: {has_trial_id.sum()} / {len(data)}")
print(f"trial_id populated but not NCT######## format: {len(bad_trial_id)}")
bad_trial_id[["_source_folder", "company", "asset_name", "trial_id"]]

trial_id populated: 107 / 925
trial_id populated but not NCT######## format: 0


,_source_folder,company,asset_name,trial_id


In [7]:
# source_url: looks like a URL
has_url = data["source_url"].notna() & (data["source_url"].astype(str).str.strip() != "")
bad_url = data[has_url & ~data["source_url"].astype(str).str.match(r"^https?://")]
print(f"source_url populated: {has_url.sum()} / {len(data)}")
print(f"source_url populated but not http(s):// : {len(bad_url)}")
bad_url[["_source_folder", "company", "asset_name", "source_url"]]

source_url populated: 925 / 925
source_url populated but not http(s):// : 0


,_source_folder,company,asset_name,source_url


In [8]:
# extraction_date: parses to a date, not in the future
parsed_date = pd.to_datetime(data["extraction_date"], errors="coerce")
has_date = data["extraction_date"].notna()
unparseable = data[has_date & parsed_date.isna()]
in_future = data[parsed_date.notna() & (parsed_date.dt.date > date.today())]

print(f"extraction_date populated: {has_date.sum()} / {len(data)}")
print(f"unparseable: {len(unparseable)}, in the future (> {date.today()}): {len(in_future)}")
print("\nextraction_date range per company:")
data.assign(extraction_date=parsed_date).groupby("_source_folder")["extraction_date"].agg(["min", "max"])

extraction_date populated: 925 / 925
unparseable: 0, in the future (> 2026-07-09): 0

extraction_date range per company:


,min,max
_source_folder,,
abbvie,2026-07-08,2026-07-08
astrazeneca,2026-07-08,2026-07-08
gsk,2026-07-08,2026-07-08
msd,2026-07-08,2026-07-08
novartis,2026-07-09,2026-07-09
novonordisk,2026-07-08,2026-07-08
pfizer,2026-07-08,2026-07-08
roche,2026-07-08,2026-07-08
sanofi,2026-07-08,2026-07-08


## 3. Sanity checks

Exact-duplicate rows, `company` value vs. its folder name, and `indication == asset_name` (known Novo Nordisk anomaly per memory — check if it recurs elsewhere).

In [9]:
# exact duplicate rows (ignoring internal _source_* meta cols)
content_cols = [c for c in data.columns if c not in META_COLS]
# list/array columns aren't hashable for duplicated() — stringify a copy for comparison only
hashable = data[content_cols].apply(lambda col: col.map(lambda v: str(v) if isinstance(v, (list, tuple, np.ndarray)) else v))
dup_mask = hashable.duplicated(keep=False)
print(f"Exact duplicate rows: {dup_mask.sum()}")
data.loc[dup_mask, ["_source_folder", "company", "asset_name", "indication", "phase"]].sort_values("_source_folder")

Exact duplicate rows: 2


,_source_folder,company,asset_name,indication,phase
188,astrazeneca,AstraZeneca,Enhertu,HER2 expressing solid tumours,Phase 3
190,astrazeneca,AstraZeneca,Enhertu,HER2 expressing solid tumours,Phase 3


In [10]:
# `company` value should be consistent within a folder (one company per file)
print("Distinct `company` values per source folder:")
print(data.groupby("_source_folder")["company"].unique().apply(list))

Distinct `company` values per source folder:
_source_folder
abbvie               [AbbVie]
astrazeneca     [AstraZeneca]
gsk                     [GSK]
msd             [Merck & Co.]
novartis           [Novartis]
novonordisk    [Novo Nordisk]
pfizer               [Pfizer]
roche                 [Roche]
sanofi               [Sanofi]
Name: company, dtype: object


In [11]:
# known anomaly type (Novo Nordisk, per memory): indication text equal to the asset_name itself
same_as_asset = data[data["indication"].astype(str).str.strip() == data["asset_name"].astype(str).str.strip()]
print(f"Rows where indication == asset_name: {len(same_as_asset)}")
same_as_asset[["_source_folder", "company", "asset_name", "indication", "therapeutic_area"]]

Rows where indication == asset_name: 0


,_source_folder,company,asset_name,indication,therapeutic_area


## 4. Coverage

Rows per company, phase distribution per company, and missing-field rate per company (optional fields legitimately vary by source richness — this is descriptive, not a pass/fail).

In [12]:
rows_per_company = data.groupby("_source_folder").size().rename("n_rows").sort_values(ascending=False)
rows_per_company

_source_folder
astrazeneca    199
roche          128
msd            109
novartis       106
abbvie          97
pfizer          96
sanofi          77
gsk             76
novonordisk     37
Name: n_rows, dtype: int64

In [13]:
phase_by_company = pd.crosstab(data["_source_folder"], data["phase"])
phase_by_company

phase,Discontinued,Phase 1,Phase 2,Phase 3,Preregistration,Registered
_source_folder,,,,,,
abbvie,0,25,30,18,6,18
astrazeneca,13,40,39,107,0,0
gsk,0,29,24,17,6,0
msd,0,0,58,35,12,4
novartis,0,23,42,38,3,0
novonordisk,0,11,12,9,5,0
pfizer,0,35,25,31,5,0
roche,0,47,17,34,6,24
sanofi,0,15,28,28,6,0


In [14]:
def is_missing(v):
    if v is None:
        return True
    if isinstance(v, float) and pd.isna(v):
        return True
    if isinstance(v, (list, tuple, np.ndarray)):
        return len(v) == 0
    if isinstance(v, str):
        return v.strip() == ""
    return False

optional_fields = ["mechanism_of_action", "therapeutic_area", "trial_id", "source_url", "extraction_date", "modality", "notes", "synonyms", "others"]
present_optional = [f for f in optional_fields if f in data.columns]

missing_rate = pd.DataFrame({
    field: data.groupby("_source_folder")[field].apply(lambda col: col.map(is_missing).mean())
    for field in present_optional
})
missing_rate.style.format("{:.0%}")

,mechanism_of_action,therapeutic_area,trial_id,source_url,extraction_date,modality,notes,synonyms,others
_source_folder,,,,,,,,,
abbvie,3%,0%,100%,0%,0%,0%,97%,100%,78%
astrazeneca,7%,7%,100%,0%,0%,7%,93%,57%,100%
gsk,0%,0%,100%,0%,0%,100%,100%,34%,0%
msd,4%,0%,2%,0%,0%,100%,8%,10%,1%
novartis,8%,0%,100%,0%,0%,100%,100%,34%,20%
novonordisk,0%,0%,100%,0%,0%,100%,100%,100%,0%
pfizer,0%,0%,100%,0%,0%,0%,100%,100%,0%
roche,32%,0%,100%,0%,0%,100%,100%,0%,0%
sanofi,0%,0%,100%,0%,0%,100%,78%,70%,77%


In [15]:
# therapeutic_area kept verbatim per-source (see memory: normalization deferred) — how fragmented is the vocab so far?
ta_counts = data.groupby("_source_folder")["therapeutic_area"].nunique().rename("n_distinct_therapeutic_area")
print(ta_counts)
print(f"\nTotal distinct therapeutic_area labels across all companies: {data['therapeutic_area'].nunique()}")
data["therapeutic_area"].value_counts().head(20)

_source_folder
abbvie         6
astrazeneca    5
gsk            4
msd            9
novartis       7
novonordisk    3
pfizer         4
roche          6
sanofi         5
Name: n_distinct_therapeutic_area, dtype: int64

Total distinct therapeutic_area labels across all companies: 34


therapeutic_area
Oncology                                    310
Immunology                                  112
Oncology/Hematology                          58
Neuroscience                                 57
Respiratory & Immunology                     28
Respiratory, Immunology and Inflammation     26
Vaccines                                     26
Cardiovascular, Renal and Metabolism         23
Infectious Diseases                          22
Cardiovascular, Renal and Metabolic          21
Inflammation & Immunology                    21
Ophthalmology                                19
Oncology: Solid Tumors                       19
Rare Disease                                 17
Diabetes                                     17
Obesity                                      14
Internal Medicine                            14
In-market Brands and Global Health           13
Rare Diseases                                13
Aesthetics                                   11
Name: count, dtype: int

## 5. Conclusions

**Overall: data is clean and schema-conformant.** 925 rows across 9 companies (AbbVie, AstraZeneca, GSK,
Merck & Co./MSD, Novartis, Novo Nordisk, Pfizer, Roche, Sanofi). Zero blank required fields, zero
out-of-enum `phase` values, zero malformed `trial_id`/`source_url`, zero unparseable or future
`extraction_date`. `company` is single-valued per file and correctly named per folder (including `msd` →
`"Merck & Co."`).

**Problems found:**

1. **2 exact duplicate rows** — AstraZeneca `Enhertu` / "HER2 expressing solid tumours" / Phase 3
   appears twice, byte-for-byte identical (rows 188, 190). Worth deduping at source or flagging in
   `astrazeneca/log.md`.
2. **`extraction_date` stored with an inconsistent physical type**: MSD's parquet stores real
   `datetime.date` objects; all 8 other companies store the same information as ISO strings. Both
   round-trip fine through `pd.to_datetime`, but downstream code doing raw type checks
   (`isinstance(v, date)`) will only work for MSD. Worth aligning MSD's converter to match the rest,
   or documenting the split.
3. **`therapeutic_area` vocabulary is fragmented, as expected/already flagged in project memory** — 34
   distinct raw labels across 9 companies for what's often the same concept (`"Oncology/Hematology"` vs.
   `"Oncology: Solid Tumors"`; `"Cardiovascular, Renal and Metabolism"` vs. `"...Metabolic"`;
   `"Respiratory & Immunology"` vs. `"Respiratory, Immunology and Inflammation"`). Confirms the
   project-memory decision to defer normalization until all companies are loaded — with 9/20 done, this
   is a good time to revisit mapping to a controlled vocabulary.
4. **Optional-field coverage is source-dependent, not a bug**: `modality` is entirely absent (no column)
   for GSK, Novo Nordisk, Roche, Sanofi; `others` absent only for AstraZeneca; `trial_id` populated for
   just 107/925 rows (12%) since it's only ever taken verbatim when a source discloses it (per
   data-model.md — never backfilled from ClinicalTrials.gov). Any downstream consumer must not assume
   these columns exist or are populated for every company.

**Checked and clean:** blank `asset_name`/`indication`/`phase`/`company`; the `indication == asset_name`
placeholder anomaly seen previously in Novo Nordisk does not recur in the other 8 companies.
**Not covered here:** transposed-column anomalies (Roche/GSK precedent) — that needs a semantic
spot-check per company, not a mechanical one, so isn't caught by this notebook.

**Reproducibility:** this notebook only reads `src/pharmas/*/*.parquet` — safe to rerun top-to-bottom
any time after new companies land; no manual steps, no writes. Re-execute headlessly with:
`uv run jupyter nbconvert --to notebook --execute --inplace notebooks/00_reviewing_results.ipynb`
(requires the `ipykernel`/`nbconvert` dev dependencies added to `pyproject.toml`).